# Biohub - Cell Tracking S1.5 Pipeline

このノートブックは、**S1.5フェーズ (スコアアップ戦略)** を実行するための統合検証ノートブックです。

## プロジェクト構成

```text
. (プロジェクトフォルダ)
├── working/                                   # 作業ディレクトリ
│   ├── s1_05_stardist_btrack_pipeline.ipynb   # main処理ノートブック
│   └── kaggle_cell_tracking_competition/      # 【準備1】主催者の公式リポジトリ
│       ├── README.md
│       ├── pyproject.toml
│       ├── tests/
│       ├── scripts/
│       └── src/
│           └── tracking_cellmot/              # 公式の評価用ライブラリ
└── input/                                     # 【準備2】inputデータ
    ├── test/                                  # 提出用データセット (.zarr / .geff)
    │   ├── xxxx.zarr/
    │   └── xxxx.geff/
    └── train/                                 # 訓練用データセット (.zarr / .geff)
        ├── xxxx.zarr/
        └── xxxx.geff/
```

### 【準備1】主催者の公式リポジトリの準備方法
working/配下でclone実行
```batch
git clone https://github.com/royerlab/kaggle_cell_tracking_competition.git
```

### 【準備2】inputデータの準備方法
`./input/` フォルダ配下に展開します。

** データの入手先:**
* [Biohub - Cell Tracking During Development Data](https://www.kaggle.com/competitions/biohub-cell-tracking-during-development/data)で、`Download All` ボタンを押し、ZIPファイルをダウンロード → 解答して展開。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

# === オフライン環境でのライブラリ自動インストール ===
import sys
import subprocess

def is_installed(package_name):
    try:
        __import__(package_name)
        return True
    except ImportError:
        return False

def run_pip(cmd_args):
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        if ipython is not None:
            ipython.system(f"pip {cmd_args}")
            return
    except ImportError:
        pass
    subprocess.run([sys.executable, "-m", "pip"] + cmd_args.split(), check=True)

# 1. Zarr パッケージのインストール
if not is_installed("zarr"):
    print("Installing zarr...")
    run_pip("install --no-index --find-links=../input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr")
else:
    print("zarr is already installed.")

# 2. tracksdata 関連パッケージのインストール
if not is_installed("tracksdata") or not is_installed("geff") or not is_installed("polars"):
    print("Installing tracksdata and dependencies...")
    run_pip("install --no-index --find-links=../input/datasets/aaaa1597/tracksdata-offline-installation-wheels rustworkx bidict ilpy imagecodecs polars")
    run_pip("install --no-index --no-deps --find-links=../input/datasets/aaaa1597/tracksdata-offline-installation-wheels geff geff-spec")
    run_pip("install --no-index --no-deps --find-links=../input/datasets/aaaa1597/tracksdata-offline-installation-wheels tracksdata")
else:
    print("tracksdata and dependencies are already installed.")

print("Offline installation steps completed.")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Offline installation check completed. (Elapsed: {_cell_elapsed:.2f}s)")


In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

import os
import sys
import glob
import numpy as np
import pandas as pd
from skimage.feature import blob_dog

# === Polarsのエラー回避モンキーパッチ ===
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32

# === パス設定 (静的解決) ===
target_path = os.path.abspath(os.path.join(os.getcwd(), '../input/datasets/aaaa1597/kaggle-cell-tracking-competition', 'src'))
if os.path.exists(os.path.join(target_path, 'tracking_cellmot')):
    if target_path not in sys.path:
        sys.path.insert(0, target_path)
    print(f"Path set successfully: {target_path}")
else:
    fallback_path = os.path.abspath(os.path.join(os.getcwd(), 'src'))
    if fallback_path not in sys.path:
        sys.path.insert(0, fallback_path)
    print(f"Warning: Expected path not found. Using fallback path: {fallback_path}")

# カスタムモジュール (trackなど) のパス登録
custom_src = os.path.abspath(os.path.join(os.getcwd(), 'src'))
if custom_src not in sys.path:
    sys.path.insert(0, custom_src)
    print(f"Added custom src to path: {custom_src}")

import tracksdata as td
from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import node_recall, evaluate, evaluate_datasets
from tracksdata.metrics import DistanceMatching
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# === GPU有無の自動判定とCPU用モンキーパッチの適用 ===
import torch
if not torch.cuda.is_available():
    print("GPU is not available. Applying CPU monkey patch to tracking_cellmot.io._process_on_gpu...")
    import tracking_cellmot.io

    def patched_process_on_gpu(
        image, tracks, scale, device,
        resample=False, target_scale=None,
        normalize=True, gamma=1.0,
        q_min=0.01, q_max=0.99, subsample_factor=1000,
        precomputed_quantiles=None
    ):
        torch_device = torch.device(device)
        image = image.astype(np.float32)

        q1, q2 = None, None
        if normalize:
            q1 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_min)
            q2 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_max)
            if q1 is None or q2 is None:
                flat = image.ravel()[::subsample_factor]
                q1, q2 = np.quantile(flat, [q_min, q_max]).astype(np.float32)
            else:
                q1 = np.float32(q1)
                q2 = np.float32(q2)

        tensor = torch.from_numpy(image)
        tensor = tensor.to(torch_device, non_blocking=True)

        if normalize:
            tensor = (tensor - float(q1)) / (float(q2) - float(q1) + 1e-6)
            tensor = tensor.clamp(min=0.0)
            if gamma != 1.0:
                tensor = tensor.pow(gamma)
            tensor = tensor.clamp(0.0, 4.0)

        if resample:
            scale_arr = np.array(scale)
            if target_scale is None:
                target_scale_val = scale_arr.min()
            else:
                target_scale_val = np.array(target_scale)

            zoom_factors = scale_arr / target_scale_val
            T = tensor.shape[0]
            new_spatial_shape = (np.array(tensor.shape[1:]) * zoom_factors).astype(int).tolist()
            tensor = tensor[:, None]
            tensor = torch.nn.functional.interpolate(
                tensor, size=new_spatial_shape, mode="trilinear", align_corners=False
            )
            tensor = tensor[:, 0]
            
            if tracks is not None:
                node_attrs = tracks.node_attrs()
                orig_dtypes = {col: node_attrs.schema[col] for col in ["z", "y", "x"]}
                node_attrs = node_attrs.with_columns(
                    (pl.col("z") * zoom_factors[0]).round(0).cast(orig_dtypes["z"]),
                    (pl.col("y") * zoom_factors[1]).round(0).cast(orig_dtypes["y"]),
                    (pl.col("x") * zoom_factors[2]).round(0).cast(orig_dtypes["x"]),
                )
                tracks.update_node_attrs(
                    attrs=node_attrs.select("z", "y", "x").to_dict(),
                    node_ids=node_attrs[td.DEFAULT_ATTR_KEYS.NODE_ID].to_list(),
                )
            if target_scale is not None:
                scale = target_scale
            else:
                scale = (float(target_scale_val),) * 3

        return tensor, tracks, scale

    tracking_cellmot.io._process_on_gpu = patched_process_on_gpu
    print("CPU Monkey Patch applied successfully!")
else:
    print("GPU is available. No monkey patch needed.")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Imports and path resolution completed. (Elapsed: {_cell_elapsed:.2f}s)")



## 【ステップ 1 & 2】検出評価と検出器の向上

In [ ]:
def detect_nodes_baseline(dataset_path, min_sigma=2.0, max_sigma=5.0, threshold=0.05, max_frames=None):
    """
    ベースライン検出器 (blob_dog)
    """
    ds = open_dataset(dataset_path, normalize=True, require_tracks=False, device='cpu')
    images = ds.image
    n_frames = images.shape[0]
    from tqdm import tqdm
    if max_frames is not None:
        n_frames = min(n_frames, max_frames)
    
    nodes = []
    global_node_id = 0
    
    for t in tqdm(range(n_frames), desc="Detecting cells in 3D frames"):
        frame = images[t]
        if hasattr(frame, 'numpy'):
            frame = frame.numpy()
        
        img_min, img_max = frame.min(), frame.max()
        if img_max > img_min:
            img_norm = (frame.astype(np.float32) - img_min) / (img_max - img_min)
        else:
            img_norm = np.zeros_like(frame, dtype=np.float32)
            
        blobs = blob_dog(img_norm, min_sigma=min_sigma, max_sigma=max_sigma, threshold=threshold)
        
        for blob in blobs:
            z, y, x, r = blob
            nodes.append({
                'node_id': global_node_id,
                't': t,
                'z': float(z),
                'y': float(y),
                'x': float(x)
            })
            global_node_id += 1
            
    return pd.DataFrame(nodes)


def evaluate_detection_only(pred_nodes_df, gt_graph, scale, max_distance=7.0):
    """
    ステップ1: 検出単体の評価 (Recall, Precision, F1-Scoreの内訳)
    """
    pred_graph = td.graph.InMemoryGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in pred_nodes_df.itertuples():
        pred_graph.add_node({
            't': int(row.t),
            'z': float(row.z),
            'y': float(row.y),
            'x': float(row.x)
        })
        
    matching = DistanceMatching(max_distance=max_distance, scale=scale)
    pred_graph.match(gt_graph, matching=matching)
    
    node_attrs = pred_graph.node_attrs(
        attr_keys=[td.DEFAULT_ATTR_KEYS.NODE_ID, td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID]
    )
    matched = node_attrs.filter(
        pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID).is_not_null()
        & (pl.col(td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID) != -1)
    )
    
    tp = matched[td.DEFAULT_ATTR_KEYS.MATCHED_NODE_ID].n_unique()
    pred_total = pred_graph.num_nodes()
    fp = pred_total - len(matched)
    
    gt_nodes_pl = gt_graph.node_attrs(attr_keys=['t'])
    gt_total_in_range = len(gt_nodes_pl.filter(pl.col('t') < 3)) if 't' in gt_nodes_pl.columns else gt_graph.num_nodes()
    fn = gt_total_in_range - tp
    
    precision = tp / pred_total if pred_total > 0 else 0.0
    recall_local = tp / gt_total_in_range if gt_total_in_range > 0 else 1.0
    recall_global = tp / gt_graph.num_nodes() if gt_graph.num_nodes() > 0 else 1.0
    
    f1_local = 2 * precision * recall_local / (precision + recall_local) if (precision + recall_local) > 0 else 0.0
    f1_global = 2 * precision * recall_global / (precision + recall_global) if (precision + recall_global) > 0 else 0.0
    
    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'pred_total': pred_total,
        'gt_total_in_range': gt_total_in_range,
        'precision': precision,
        'recall_local': recall_local,
        'recall_global': recall_global,
        'f1_local': f1_local,
        'f1_global': f1_global
    }

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Cell Cell detection and evaluation functions defined. (Elapsed: {_cell_elapsed:.2f}s)")


### 1.1 検出結果の3D視覚化 (Error Analysis)

予測された細胞位置 (ノード) と GT を3D空間にプロットし、公式マッチング基準 (7 µm) で正しく検出できたもの (TP: 緑)、余分な検出 (FP: 赤)、見落とした正解 (FN: 青) を色分けして可視化します。

In [ ]:
# === Interactive 3D MIP Overlays Dashboard Definition (Camera-Synchronized) ===
import plotly.graph_objects as go
import plotly.offline as pyo
from plotly.subplots import make_subplots
from scipy.spatial import KDTree
import numpy as np
import pandas as pd
import re
from IPython.display import HTML, display

def plot_detection_dashboard(image_3d, pred_nodes_df, gt_graph, t_selected=0, max_distance=7.0, scale=[1.0, 1.0, 1.0]):
    """
    指定したフレームにおける検出結果を 2x2 の 3D インタラクティブサブプロットで可視化します。
    (1,1): 3D Point Cloud (MIPなし、3D空間に浮遊)
    (1,2): XY-MIP 3D Overlay (Z=0.0 底面にMIP画像と投影点群を貼り付け)
    (2,1): XZ-MIP 3D Overlay (Y=0.0 背面にMIP画像と投影点群を貼り付け)
    (2,2): YZ-MIP 3D Overlay (X=0.0 側面にMIP画像と投影点群を貼り付け)
    """
    scale = np.array(scale)
    nz, ny, nx = image_3d.shape
    
    # 各軸方向のMIP画像を作成
    mip_z = np.max(image_3d, axis=0)  # XY平面のMIP
    mip_y = np.max(image_3d, axis=1)  # XZ平面のMIP
    mip_x = np.max(image_3d, axis=2)  # YZ平面のMIP
    
    # 物理座標グリッドの生成 (基準壁面 Z=0.0, Y=0.0, X=0.0 に配置)
    x_grid_z, y_grid_z = np.meshgrid(np.arange(nx) * scale[2], np.arange(ny) * scale[1])
    z_surface_z = np.zeros_like(mip_z)  # これは背景画像なので Z=0.0 で配置します
    
    x_grid_y, z_grid_y = np.meshgrid(np.arange(nx) * scale[2], np.arange(nz) * scale[0])
    y_surface_y = np.zeros_like(x_grid_y)  # これは背景画像なので Y=0.0 で配置します
    
    y_grid_x, z_grid_x = np.meshgrid(np.arange(ny) * scale[1], np.arange(nz) * scale[0])
    x_surface_x = np.zeros_like(y_grid_x)  # これは背景画像なので X=0.0 で配置します

    # GTの座標抽出
    gt_nodes_pl = gt_graph.node_attrs(attr_keys=['t', 'z', 'y', 'x'])
    gt_nodes_df = gt_nodes_pl.to_pandas()
    
    p_nodes = pred_nodes_df[pred_nodes_df['t'] == t_selected].copy()
    g_nodes = gt_nodes_df[gt_nodes_df['t'] == t_selected].copy()
    
    p_coords = p_nodes[['z', 'y', 'x']].values * scale
    g_coords = g_nodes[['z', 'y', 'x']].values * scale
    
    tp_p_idx, tp_g_idx = [], []
    if len(p_coords) > 0 and len(g_coords) > 0:
        tree = KDTree(g_coords)
        distances, indices = tree.query(p_coords, distance_upper_bound=max_distance)
        
        matched_g = set()
        for p_idx, (d, g_idx) in enumerate(zip(distances, indices)):
            if d <= max_distance and g_idx not in matched_g:
                tp_p_idx.append(p_idx)
                tp_g_idx.append(g_idx)
                matched_g.add(g_idx)
                
    tp_coords = p_coords[tp_p_idx] if tp_p_idx else np.empty((0, 3))
    fp_coords = np.delete(p_coords, tp_p_idx, axis=0) if len(p_coords) > 0 else np.empty((0, 3))
    fn_coords = np.delete(g_coords, tp_g_idx, axis=0) if len(g_coords) > 0 else np.empty((0, 3))
    
    # 各MIP投影面での点群の浮き上がりオフセット量を定義します
    proj_z = 0.1
    proj_y = 0.1
    proj_x = 0.1
    
    fig = make_subplots(
        rows=2, cols=2,
        specs=[
            [{"type": "scene"}, {"type": "scene"}],
            [{"type": "scene"}, {"type": "scene"}]
        ],
        subplot_titles=(
            "1. 3D Point Cloud (Interactive TP/FP/FN)",
            "2. XY-MIP 3D Overlay (Z=0.0 Bottom Wall)",
            "3. XZ-MIP 3D Overlay (Y=0.0 Back Wall)",
            "4. YZ-MIP 3D Overlay (X=0.0 Side Wall)"
        )
    )
    
    if len(tp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=tp_coords[:, 2], y=tp_coords[:, 1], z=tp_coords[:, 0],
            mode='markers', name='TP (Correct)',
            marker=dict(size=4, color='lime', opacity=0.8, symbol='circle'),
            showlegend=True
        ), row=1, col=1)
    if len(fp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fp_coords[:, 2], y=fp_coords[:, 1], z=fp_coords[:, 0],
            mode='markers', name='FP (Extra)',
            marker=dict(size=4, color='red', opacity=0.8, symbol='x'),
            showlegend=True
        ), row=1, col=1)
    if len(fn_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fn_coords[:, 2], y=fn_coords[:, 1], z=fn_coords[:, 0],
            mode='markers', name='FN (Missing)',
            marker=dict(size=4, color='cyan', opacity=0.8, symbol='diamond'),
            showlegend=True
        ), row=1, col=1)
    
    # XY-MIP (Z=0.0 面の上に proj_z 浮かせる)
    fig.add_trace(go.Surface(
        x=x_grid_z, y=y_grid_z, z=z_surface_z,
        surfacecolor=mip_z, colorscale='Gray', showscale=False,
        hoverinfo='x+y+z', name='MIP Z'
    ), row=1, col=2)
    if len(tp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=tp_coords[:, 2], y=tp_coords[:, 1], z=np.full(len(tp_coords), proj_z),
            mode='markers', name='TP (XY)',
            marker=dict(size=4, color='lime', opacity=0.9, symbol='circle'), showlegend=False
        ), row=1, col=2)
    if len(fp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fp_coords[:, 2], y=fp_coords[:, 1], z=np.full(len(fp_coords), proj_z),
            mode='markers', name='FP (XY)',
            marker=dict(size=4, color='red', opacity=0.9, symbol='x'), showlegend=False
        ), row=1, col=2)
    if len(fn_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fn_coords[:, 2], y=fn_coords[:, 1], z=np.full(len(fn_coords), proj_z),
            mode='markers', name='FN (XY)',
            marker=dict(size=4, color='cyan', opacity=0.9, symbol='diamond'), showlegend=False
        ), row=1, col=2)
    
    # XZ-MIP (Y=0.0 面の上に proj_y 浮かせる)
    fig.add_trace(go.Surface(
        x=x_grid_y, y=y_surface_y, z=z_grid_y,
        surfacecolor=mip_y, colorscale='Gray', showscale=False,
        hoverinfo='x+y+z', name='MIP Y'
    ), row=2, col=1)
    if len(tp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=tp_coords[:, 2], y=np.full(len(tp_coords), proj_y), z=tp_coords[:, 0],
            mode='markers', name='TP (XZ)',
            marker=dict(size=4, color='lime', opacity=0.9, symbol='circle'), showlegend=False
        ), row=2, col=1)
    if len(fp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fp_coords[:, 2], y=np.full(len(fp_coords), proj_y), z=fp_coords[:, 0],
            mode='markers', name='FP (XZ)',
            marker=dict(size=4, color='red', opacity=0.9, symbol='x'), showlegend=False
        ), row=2, col=1)
    if len(fn_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=fn_coords[:, 2], y=np.full(len(fn_coords), proj_y), z=fn_coords[:, 0],
            mode='markers', name='FN (XZ)',
            marker=dict(size=4, color='cyan', opacity=0.9, symbol='diamond'), showlegend=False
        ), row=2, col=1)
    
    # YZ-MIP (X=0.0 面の上に proj_x 浮かせる)
    fig.add_trace(go.Surface(
        x=x_surface_x, y=y_grid_x, z=z_grid_x,
        surfacecolor=mip_x, colorscale='Gray', showscale=False,
        hoverinfo='x+y+z', name='MIP X'
    ), row=2, col=2)
    if len(tp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=np.full(len(tp_coords), proj_x), y=tp_coords[:, 1], z=tp_coords[:, 0],
            mode='markers', name='TP (YZ)',
            marker=dict(size=4, color='lime', opacity=0.9, symbol='circle'), showlegend=False
        ), row=2, col=2)
    if len(fp_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=np.full(len(fp_coords), proj_x), y=fp_coords[:, 1], z=fp_coords[:, 0],
            mode='markers', name='FP (YZ)',
            marker=dict(size=4, color='red', opacity=0.9, symbol='x'), showlegend=False
        ), row=2, col=2)
    if len(fn_coords) > 0:
        fig.add_trace(go.Scatter3d(
            x=np.full(len(fn_coords), proj_x), y=fn_coords[:, 1], z=fn_coords[:, 0],
            mode='markers', name='FN (YZ)',
            marker=dict(size=4, color='cyan', opacity=0.9, symbol='diamond'), showlegend=False
        ), row=2, col=2)
    
    scene_config = dict(
        xaxis=dict(title='X (Width, µm)', range=[0, nx * scale[2]]),
        yaxis=dict(title='Y (Height, µm)', range=[0, ny * scale[1]]),
        zaxis=dict(title='Z (Depth, µm)', range=[0, nz * scale[0]]),
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=0.6),
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    )
    scene2_config = scene_config.copy()
    scene2_config['camera'] = dict(eye=dict(x=1.2, y=1.2, z=1.5))
    
    fig.update_layout(
        title_text=f"Cell Detection Interactive 3D MIP Overlays Dashboard (Frame {t_selected})",
        scene1=scene_config,
        scene2=scene2_config,
        scene3=scene_config,
        scene4=scene_config,
        autosize=True,
        height=950,
        margin=dict(l=20, r=20, b=20, t=80)
    )
    
    plot_div = pyo.plot(fig, include_plotlyjs=True, output_type='div')
    match = re.search(r'id="([^"]+)"', plot_div)
    if match:
        plot_id = match.group(1)
        sync_js = """
        <script>
        (function() {
            var gd = document.getElementById('__PLOT_ID__');
            var scenes = ['scene', 'scene2', 'scene3', 'scene4'];
            var updating = false;
            
            gd.on('plotly_relayout', function(eventdata) {
                if (updating) return;
                
                var camera = null;
                var changedScene = null;
                
                for (var i = 0; i < scenes.length; i++) {
                    var sceneKey = scenes[i] + '.camera';
                    if (eventdata[sceneKey] !== undefined) {
                        camera = eventdata[sceneKey];
                        changedScene = scenes[i];
                        break;
                    }
                }
                
                if (camera !== null) {
                    updating = true;
                    var update = {};
                    for (var i = 0; i < scenes.length; i++) {
                        if (scenes[i] !== changedScene) {
                            update[scenes[i] + '.camera'] = camera;
                        }
                    }
                    Plotly.relayout(gd, update).then(function() {
                        updating = false;
                    });
                }
            });
        })();
        </script>
        """.replace('__PLOT_ID__', plot_id)
        display(HTML(plot_div + sync_js))
    else:
        display(HTML(plot_div))


## 【ステップ 3 & 4】トラッキング評価とトラッカーの向上

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

def evaluate_tracking_only(pred_edges, gt_graph, gt_nodes_df, scale):
    """
    ステップ3: トラッキング単体の評価 (GTノードを直接入力)
    """
    from tracking_cellmot.metrics import evaluate
    from tracksdata.graph import IndexedRXGraph
    
    pred_graph = IndexedRXGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in gt_nodes_df.itertuples():
        pred_graph.add_node(
            attrs={
                't': int(row.t),
                'z': float(row.z),
                'y': float(row.y),
                'x': float(row.x)
            },
            index=int(row.node_id)
        )
        
    for edge in pred_edges:
        pred_graph.add_edge(int(edge['source_id']), int(edge['target_id']), attrs={})
        
    res = evaluate(pred_graph, gt_graph, scale=scale)
    
    edge_denom = res.edge_tp + res.edge_fp + res.edge_fn
    edge_jaccard = res.edge_tp / edge_denom if edge_denom > 0 else 1.0
    
    return {
        'edge_jaccard': edge_jaccard,
        'edge_tp': res.edge_tp,
        'edge_fp': res.edge_fp,
        'edge_fn': res.edge_fn
    }

print("Tracking evaluation functions defined.")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Cell execution completed. (Elapsed: {_cell_elapsed:.2f}s)")


## 【ステップ 5 & 6】統合評価と間引き (Pruning) の実行

検出予測ノードを用いたトラッキングの実行と公式評価（統合評価）、および不要なノードやエッジを削る間引き (Pruning) 処理を行います。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

def evaluate_complete(pred_nodes_df, pruned_edges, gt_graph, scale):
    """
    ステップ5: 統合評価 (検出予測ノードを用いたトラッキング実行と公式評価)
    """
    from tracking_cellmot.metrics import evaluate
    from tracksdata.graph import IndexedRXGraph
    
    pred_graph = IndexedRXGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)
        
    for row in pred_nodes_df.itertuples():
        pred_graph.add_node(
            attrs={
                't': int(row.t),
                'z': float(row.z),
                'y': float(row.y),
                'x': float(row.x)
            },
            index=int(row.node_id)
        )
        
    for edge in pruned_edges:
        pred_graph.add_edge(int(edge['source_id']), int(edge['target_id']), attrs={})
        
    res = evaluate(pred_graph, gt_graph, scale=scale)
    return res, pred_graph

def prune_tracks(nodes_df, edges):
    """
    ステップ6: 間引き (Pruning) / 後処理の実行（プレースホルダー）
    """
    print("Pruning placeholder: returning input nodes and edges without modifications.")
    return nodes_df, edges

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Complete evaluation and pruning functions defined. (Elapsed: {_cell_elapsed:.2f}s)")


## ベースライン全体の実行とテスト

データセットを読み込み、これまでに実装した検出器の検証を行います。

In [ ]:
# === 実行時間計測の開始 ===
import time
_cell_start_time = time.time()

# データ読み込みとmain()テスト実行
import sys
import glob
import zarr
import numpy as np
import pandas as pd
from tracking_cellmot.io import open_dataset
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'input', 'train'))
dataset_paths = glob.glob(os.path.join(DATA_DIR, '*.zarr')) + glob.glob(os.path.join(DATA_DIR, '*.geff'))
if not dataset_paths:
    dataset_paths = glob.glob('/kaggle/input/**/train/*.zarr', recursive=True) + glob.glob('/kaggle/input/**/train/*.geff', recursive=True)

if not dataset_paths:
    raise FileNotFoundError("Error: No .zarr or .geff datasets found in the data directories.")

target_dataset_path = dataset_paths[0]
print(f'Target dataset: {target_dataset_path}')

# GTのロード (失敗時はエラー終了)
try:
    ds_gt = open_dataset(target_dataset_path, normalize=True, require_tracks=True, device='cpu')
except Exception as e:
    print(f"Error: Failed to load dataset {target_dataset_path}. Reason: {e}", file=sys.stderr)
    raise e

gt_graph = ds_gt.tracks
scale = ds_gt.scale

# 1. 検出評価
print('\n--- Running Node Detection Evaluation ---')
pred_nodes = detect_nodes_baseline(target_dataset_path, max_frames=3)  # トラッキング評価用に3フレーム実行
det_res = evaluate_detection_only(pred_nodes, gt_graph, scale=scale)

print("\n=================== DETAILED DETECTION EVALUATION ===================")
print(f"  TP (Matched predicted nodes): {det_res['tp']}")
print(f"  FP (Extra predicted nodes):   {det_res['fp']}")
print(f"  FN (Missed GT nodes in range):{det_res['fn']} (GT in range total: {det_res['gt_total_in_range']})")
print(f"  Detection Precision:          {det_res['precision']:.4f}  [ Formula: TP / T_pred = {det_res['tp']} / {det_res['pred_total']} ]")
print(f"  Detection Recall (Local):     {det_res['recall_local']:.4f}  [ Formula: TP / GT_in_range = {det_res['tp']} / {det_res['gt_total_in_range']} ]")
print(f"  Detection Recall (Global):    {det_res['recall_global']:.4f}  [ Formula: TP / GT_total = {det_res['tp']} / {gt_graph.num_nodes()} ]")
print(f"  Detection F1-Score (Local):   {det_res['f1_local']:.4f}")
print(f"  Detection F1-Score (Global):  {det_res['f1_global']:.4f}")
print("=====================================================================\n")

# 可視化の実行 (MIP画像の取得とダッシュボード表示)
t_selected = 0  # 可視化対象の時間フレームを選択
print(f'\n--- Visualizing frame {t_selected} cell centroids ---')
img_3d = ds_gt.image[t_selected]
if hasattr(img_3d, 'numpy'):
    img_3d = img_3d.numpy()
plot_detection_dashboard(img_3d, pred_nodes, gt_graph, t_selected=t_selected, scale=scale)

# 2. トラッキング評価 (GTノードを入力)
print('\n--- Running Tracking Evaluation with GT Nodes ---')
from track.btrack_tracker import run_btrack_tracking
gt_nodes_pl = gt_graph.node_attrs(attr_keys=['node_id', 't', 'z', 'y', 'x'])
gt_nodes_df = gt_nodes_pl.to_pandas()

# btrack（またはNN）実行
edges = run_btrack_tracking(gt_nodes_df, max_search_radius=25.0)
track_res = evaluate_tracking_only(edges, gt_graph, gt_nodes_df, scale=scale)
print(f'Edge Jaccard (GT Nodes): {track_res["edge_jaccard"]:.4f} (TP={track_res["edge_tp"]}, FP={track_res["edge_fp"]}, FN={track_res["edge_fn"]})')

# 3. 統合評価 (検出予測ノードを用いたトラッキング実行)
print('\n--- Running Complete End-to-End Evaluation ---')
complete_edges = run_btrack_tracking(pred_nodes, max_search_radius=25.0)

# 4. 間引き (Pruning) / 後処理の実行
pruned_nodes, pruned_edges = prune_tracks(pred_nodes, complete_edges)

# 5. 統合スコア評価とペナルティ（Adjusted Jaccard）の算出
complete_res, pred_graph = evaluate_complete(pruned_nodes, pruned_edges, gt_graph, scale=scale)

# OME-NGFF Zarrの属性から estimated_number_of_nodes の取得を試みる
try:
    img_ds = zarr.open_group(target_dataset_path, mode="r")
    n_total_estimated = float(img_ds.attrs.get("estimated_number_of_nodes", float('nan')))
except Exception as e:
    print(f"Warning: Could not read estimated_number_of_nodes from metadata: {e}")
    n_total_estimated = float('nan')

# なければ、この3フレームにおけるGTノード数（スパース）でフォールバック
if np.isnan(n_total_estimated):
    n_total_estimated = gt_graph.num_nodes()

from tracking_cellmot.metrics import per_sample_metrics, node_recall
rec_val = node_recall(pred_graph, gt_graph)
p_metrics = per_sample_metrics(complete_res, n_total=n_total_estimated, node_recall=rec_val)

print("\n=================== INTEGRATED EVALUATION RESULTS ===================")
print(f"  Estimated True Nodes (n_total, Dataset Total GT): {n_total_estimated}")
print(f"  Predicted Nodes (T_pred, 3-Frame Total):          {p_metrics['num_pred_nodes']}")
print(f"  Node Recall (matched ratio):                      {p_metrics['node_recall']:.4f}  [ Matched ({det_res['tp']}) / Dataset Total GT ({int(n_total_estimated)}) ]")

print(f"  Edge Jaccard:                                     {p_metrics['edge_jaccard']:.4f}  (TP={p_metrics['edge_tp']}, FP={p_metrics['edge_fp']}, FN={p_metrics['edge_fn']})")
print(f"    [ Formula: TP / (TP + FP + FN) = {p_metrics['edge_tp']} / ({p_metrics['edge_tp']} + {p_metrics['edge_fp']} + {p_metrics['edge_fn']}) ]")

print(f"  Node Ratio (extra nodes ratio):                  {p_metrics['total_node_ratio']:.4f}")
print(f"    [ Formula: (T_pred - n_total) / n_total = ({p_metrics['num_pred_nodes']} - {int(n_total_estimated)}) / {int(n_total_estimated)} ]")

ratio_val = p_metrics['total_node_ratio']
penalty_val = 1.0 / max(1.0, ratio_val - 3.0)
print(f"  Penalty Coefficient (P):                          {penalty_val:.4f}")
print(f"    [ Formula: 1.0 / max(1.0, Ratio - 3.0) = 1.0 / ({ratio_val:.4f} - 3.0) ]")

print(f"  Adjusted Edge Jaccard:                            {p_metrics['adj_edge_jaccard']:.4f}")
print(f"    [ Formula: Edge Jaccard ({p_metrics['edge_jaccard']:.4f}) * Penalty ({penalty_val:.4f}) -> Adjusted ]")

div_denom = p_metrics['division_tp'] + p_metrics['division_fp'] + p_metrics['division_fn']
div_jaccard = p_metrics['division_tp'] / div_denom if div_denom > 0 else 0.0
print(f"  Division Jaccard:                                 {div_jaccard:.4f}  (TP={p_metrics['division_tp']}, FP={p_metrics['division_fp']}, FN={p_metrics['division_fn']})")
print(f"    [ Formula: TP / (TP + FP + FN) ]")

print(f"  COMBINED SCORE:                                   {p_metrics['adj_edge_jaccard'] + 0.1 * div_jaccard:.4f}")
print(f"    [ Formula: Adjusted Edge Jaccard ({p_metrics['adj_edge_jaccard']:.4f}) + 0.1 * Division Jaccard ({div_jaccard:.4f}) ]")
print("=====================================================================\n")

# === 実行時間計測の終了 ===
import datetime
_cell_elapsed = time.time() - _cell_start_time
_jst_tz = datetime.timezone(datetime.timedelta(hours=9))  # 日本時間 (UTC+9)
_cell_current_time = datetime.datetime.now(_jst_tz).strftime("%Y-%m-%d %H:%M:%S")
print(f"\n[{_cell_current_time} JST] Pipeline detection evaluation completed. (Elapsed: {_cell_elapsed:.2f}s)")
